In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import EMNIST
from torch.utils.data import DataLoader

# Define transforms - COMPLETE THE MISSING PARTS
transform = transforms.Compose([
    # TODO: Resize to 28x28
    transforms.Resize((28, 28)),
    transforms.Grayscale(3),  # Convert grayscale to RGB (Don't Touch!!)
    # TODO: Convert to Tensor
    transforms.ToTensor(),
    # TODO: Normalize with ImageNet mean=[0.485, 0.456, 0.406] and std=[0.229, 0.224, 0.225]
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                          std=[0.229, 0.224, 0.225])
])

# Load EMNIST letters dataset (given)
train_dataset = EMNIST(root='./data', split='letters', train=True, download=True, transform=transform)
test_dataset = EMNIST(root='./data', split='letters', train=False, download=True, transform=transform)

# data loader
train_loader = DataLoader(train_dataset, batch_size=10, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=10, shuffle=False)

# Note: EMNIST letters has labels 1-26 (A-Z), so we have 26 classes
num_classes = 26

print(f"Training samples: {len(train_dataset)}")
print(f"Testing samples: {len(test_dataset)}")
print(f"Number of classes: {num_classes}")

In [ ]:
# Letter mapping (labels are 1-26 for A-Z)
letters = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'

# Create DataLoaders and display samples
# Write your code here

#DataLoaders
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)


# Create a function to visualize samples
import matplotlib.pyplot as plt
import random
import numpy as np

def visualize_samples(dataset, num_samples=8, title="Dataset Samples"):
    """
    Visualize random samples from a dataset.

    Args:
        dataset: PyTorch Dataset object
        num_samples: Number of samples to display
        title: Title for the plot
    """
    # Select random indices
    indices = random.sample(range(len(dataset)), num_samples)

    # Calculate grid size
    cols = 4
    rows = (num_samples + cols - 1) // cols

    # Create the plot
    fig, axes = plt.subplots(rows, cols, figsize=(12, 3 * rows))
    axes = axes.flatten()

    for i, idx in enumerate(indices):
        # Get image and label
        image, label = dataset[idx]
        # Convert tensor to numpy for display
        if isinstance(image, torch.Tensor):
            image = image.permute(1, 2, 0).numpy()  # CHW -> HWC
        # Get class name
        class_name = dataset.classes[label]
        # Display image
        axes[i].imshow(image)
        axes[i].set_title(f"{class_name}\n(Label: {label})", fontsize=10)
        axes[i].axis('off')
    # Hide any unused subplots
    for i in range(num_samples, len(axes)):
        axes[i].axis('off')

    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()


visualize_samples(train_dataset)

In [ ]:
import torch.nn as nn
from torchvision.models import efficientnet_v2_s

# Write your code here

# loading the model
model = efficientnet_v2_s(pretrained=True)

# freeze all then activate the head
model.requires_grad_(False)
model.classifier.requires_grad_(True)

# adjust the head to out 26
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 26)

# confirm
model

In [ ]:
# Write your code here

# Training and Validation Loops
def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train() # Set the model to training mode
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    for inputs, labels in dataloader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad() # Zero the parameter gradients

        outputs = model(inputs)
        loss = criterion(outputs, (labels-1))
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_samples += labels.size(0)
        correct_predictions += (predicted == labels).sum().item()

    epoch_loss = running_loss / total_samples
    epoch_accuracy = correct_predictions / total_samples
    return epoch_loss, epoch_accuracy

def validate_epoch(model, dataloader, criterion, device):
    model.eval() # Set the model to evaluation mode
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    with torch.no_grad(): # Disable gradient calculation
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, (labels-1))

            running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total_samples += labels.size(0)
            correct_predictions += (predicted == labels).sum().item()

    epoch_loss = running_loss / total_samples
    epoch_accuracy = correct_predictions / total_samples
    return epoch_loss, epoch_accuracy

In [ ]:
# Write your code here
import torch
import torch.nn as nn
import torch.optim as optim

# set up the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


criterion = nn.CrossEntropyLoss() #                       (the loss funcation)
optimizer = optim.Adam(model.parameters(), lr=0.001) #.   (the optinizer)


num_epochs = 3

train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

print("Starting Training...")
for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate_epoch(model, test_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)

    print(f'Epoch {epoch+1}/{num_epochs} - Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}')

print("Finished Training.")

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(range(1, num_epochs + 1), train_losses, label='Training Loss')
plt.plot(range(1, num_epochs + 1), val_losses, label='Validation Loss')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(range(1, num_epochs + 1), train_accuracies, label='Training Accuracy')
plt.plot(range(1, num_epochs + 1), val_accuracies, label='Validation Accuracy')
plt.title('Accuracy over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Write your code here
